# KubeBalancer on FABRIC: a 3-node k3s cluster

This notebook builds the Milestone 2 cluster on FABRIC and deploys KubeBalancer on it:

1. Create a slice with three Ubuntu VMs at one site, joined by a private L2 network (`192.168.10.0/24`).
2. Give each VM its address on that network and make sure it can reach the internet (for `get.k3s.io` and `ghcr.io`).
3. Install k3s: `k3s-server` runs the control plane, `k3s-agent-1` and `k3s-agent-2` join it.
4. Deploy the manifests from the repository and run the smoke test from inside the cluster.
5. Print the SSH tunnel command for reaching the gateway (NodePort 30080) from your laptop.
6. Run `scripts/evidence.sh fabric` on the server and download the evidence.

Run it in the FABRIC JupyterHub, from the `fabric/` folder of the repository, so `install-k3s.sh` is next to it.
The images must already be on GHCR and public (`scripts/build-images.sh --push`; see `fabric/README.md`).

In [ ]:
import os
import secrets
import tempfile
from ipaddress import IPv4Address, IPv4Network, ip_address

from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()
fablib.show_config();

## Settings

`SITE = None` picks a random site with free capacity. Prefer a site whose VMs get an **IPv4**
management address. On IPv6-only sites, `USE_PUBLIC_NAT64 = True` points DNS at the public
nat64.net resolvers so the VMs can reach IPv4-only hosts such as `ghcr.io`. That makes a third
party responsible for name resolution and for carrying the traffic. Downloads stay
integrity-protected (HTTPS with certificate checks, plus the pinned k3s installer checksum), but
the operator can see which hosts the VMs look up and could block them. Set it to `False` to
refuse, and use another site or `fabric/load-images.sh` instead.

In [ ]:
SLICE_NAME = "kubebalancer-m2"
SITE = None                      # e.g. "TACC", "STAR", "UTAH"; None = random site with capacity
IMAGE = "default_ubuntu_24"
CORES, RAM_GB, DISK_GB = 4, 8, 20

NETWORK = "cluster-net"
SUBNET = IPv4Network("192.168.10.0/24")
NODES = {                        # name -> address on the private network; the server comes first
    "k3s-server": "192.168.10.10",
    "k3s-agent-1": "192.168.10.11",
    "k3s-agent-2": "192.168.10.12",
}
SERVER = "k3s-server"
SERVER_IP = NODES[SERVER]

USE_PUBLIC_NAT64 = True          # only used on IPv6-only sites; see above

REPO_URL = "https://github.com/angeudybp/csc478-project.git"
REPO_DIR = "kubebalancer"

## 1. Create the slice

In [ ]:
site = SITE or fablib.get_random_site()
print(f"Creating {SLICE_NAME} at {site}")

slice = fablib.new_slice(name=SLICE_NAME)
interfaces = []
for name in NODES:
    node = slice.add_node(name=name, site=site, cores=CORES, ram=RAM_GB, disk=DISK_GB, image=IMAGE)
    interfaces.append(node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0])
slice.add_l2network(name=NETWORK, interfaces=interfaces, subnet=SUBNET)

slice.submit();

## 2. Configure the private network and internet access

Addresses added with `ip_addr_add` do not survive a VM reboot; rerun this cell if a VM restarts.

In [ ]:
slice = fablib.get_slice(name=SLICE_NAME)

NAT64_DNS = ["2a00:1098:2c::1", "2a01:4f8:c2c:123f::1"]  # public NAT64/DNS64 resolvers (nat64.net)

for name, ip in NODES.items():
    node = slice.get_node(name=name)
    iface = node.get_interface(network_name=NETWORK)
    iface.ip_addr_add(addr=IPv4Address(ip), subnet=SUBNET)
    iface.ip_link_up()

    mgmt = node.get_management_ip()
    if ip_address(mgmt).version == 6 and USE_PUBLIC_NAT64:
        # IPv6-only management network: resolve IPv4-only hosts through DNS64 so NAT64 can reach them.
        resolv = "\\n".join(f"nameserver {s}" for s in NAT64_DNS)
        node.execute(f"printf '{resolv}\\n' | sudo tee /etc/resolv.conf.nat64 >/dev/null && "
                     "sudo ln -sf /etc/resolv.conf.nat64 /etc/resolv.conf", quiet=True)

    out, _ = node.execute("curl -s -o /dev/null -w '%{http_code}' https://ghcr.io/v2/", quiet=True)
    print(f"{name:12} mgmt {mgmt:40} {iface.get_os_interface():10} {ip:15} ghcr.io -> HTTP {out.strip()}")

Every node should report `ghcr.io -> HTTP 401` (reachable; 401 just means "no login", which is fine for
public images). `000` means the VM cannot reach GHCR: pick another site, or load the images with
`fabric/load-images.sh` after step 3.

Check that the nodes can reach each other on the private network:

In [ ]:
server = slice.get_node(name=SERVER)
for name, ip in NODES.items():
    if name != SERVER:
        out, _ = server.execute(f"ping -c 2 -W 2 {ip} | tail -1", quiet=True)
        print(f"{SERVER} -> {name} ({ip}): {out.strip()}")

## 3. Install k3s (server first, then the agents)

Tokens are copied to the VMs as files over SFTP, never placed on a command line, where they would
show up in `ps` and in the sudo log. The agents join with the server's `node-token`, which
contains the cluster CA hash, so each agent checks that it is talking to this server.

In [ ]:
def upload_secret(node, value, remote_path):
    """Copy a secret to the VM over SFTP, so it never appears in a command line or log."""
    fd, local_path = tempfile.mkstemp()
    try:
        with os.fdopen(fd, "w") as f:
            f.write(value)
        node.upload_file(local_path, remote_path)
    finally:
        os.remove(local_path)
    node.execute(f"chmod 600 {remote_path}", quiet=True)


def install(node, args):
    node.upload_file("install-k3s.sh", "install-k3s.sh")
    out, err = node.execute(f"chmod +x install-k3s.sh && sudo ./install-k3s.sh {args}", quiet=True)
    print(f"--- {node.get_name()}\n{out[-600:]}{err[-600:]}")


server = slice.get_node(name=SERVER)
upload_secret(server, secrets.token_urlsafe(32), "k3s-token")
install(server, f"server {SERVER_IP} k3s-token")

node_token, _ = server.execute("sudo cat /var/lib/rancher/k3s/server/node-token", quiet=True)
for name, ip in NODES.items():
    if name != SERVER:
        agent = slice.get_node(name=name)
        upload_secret(agent, node_token.strip(), "k3s-token")
        install(agent, f"agent {ip} k3s-token {SERVER_IP}")

In [ ]:
# All three nodes should be Ready, each with its 192.168.10.x address as INTERNAL-IP.
server.execute("for i in $(seq 30); do [ $(kubectl get nodes --no-headers | grep -c ' Ready') -eq 3 ] && break; sleep 5; done; "
               "kubectl get nodes -o wide");

## 4. Deploy KubeBalancer

In [ ]:
server.execute(f"[ -d {REPO_DIR} ] && git -C {REPO_DIR} pull --ff-only || git clone {REPO_URL} {REPO_DIR}")
server.execute(f"cd {REPO_DIR} && kubectl apply -f deployments/ -f services/ && "
               "for d in gateway preprocess inference storage; do kubectl -n imgpipe rollout status deploy/$d --timeout=300s; done && "
               "kubectl -n imgpipe get deploy,pods,svc -o wide");

The inference pods should be spread over more than one node (NODE column). Now the smoke test,
through the NodePort on the server's private address:

In [ ]:
server.execute(f"cd {REPO_DIR} && GATEWAY_URL=http://{SERVER_IP}:30080 scripts/smoke.sh");

## 5. Reach the gateway from your laptop

FABRIC VMs sit behind a bastion host, so the gateway is reached through an SSH tunnel to the
NodePort. Run the printed command on your laptop (with your FABRIC `ssh_config` and keys, see
`fabric/README.md`), keep it open, then browse to http://localhost:30080 or run
`GATEWAY_URL=http://localhost:30080 scripts/smoke.sh`.

In [ ]:
ssh_cmd = server.get_ssh_command()
print(ssh_cmd.replace("ssh ", f"ssh -N -L 30080:{SERVER_IP}:30080 ", 1))

## 6. Capture the Milestone 2 evidence

`scripts/evidence.sh` runs on the server (it has `kubectl`, `curl` and `python3`) and writes to
`docs/evidence/m2/fabric/`. The last step deletes and re-applies the whole stack. Your laptop's
SSH tunnel keeps working, because the NodePort stays 30080. Take terminal screenshots while it runs
for the README.

In [ ]:
out, err = server.execute(f"cd {REPO_DIR} && GATEWAY_URL=http://{SERVER_IP}:30080 scripts/evidence.sh fabric "
                          "> evidence-run.log 2>&1; echo exit=$?; tail -5 evidence-run.log", quiet=True)
print(out, err)

server.execute(f"tar czf evidence-fabric.tgz -C {REPO_DIR}/docs/evidence/m2 fabric", quiet=True)
server.download_file("evidence-fabric.tgz", "evidence-fabric.tgz")
print("Downloaded evidence-fabric.tgz: unpack it into docs/evidence/m2/ in your repository and commit it.")

## 7. Delete the slice when you are done

Slices expire on their own, but deleting frees the resources for others. Everything needed to
rebuild is in this notebook.

In [ ]:
# slice.delete()